# SigExt Intrinsic Evaluation: Ranking the Extractors

This notebook evaluates the standalone performance of all trained **SigExt** models. It computes token-level metrics (**Precision, Recall, F1**) against Sentence-BERT "gold" labels on a held-out test set. 

This allows us to identify which configurations (base model, threshold, sample size) are best at identifying salient sentences **intrinsicly**, without involving the LLM decoder.

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.pipelines import run_sigext_evaluation_matrix
import pandas as pd

# 1. Load all configurations from the presets
configs = SigExtConfig.get_full_matrix()

# 2. Run evaluation on a held-out test set (500 samples)
# This will compute metrics for all 27 models
results = run_sigext_evaluation_matrix(configs, test_samples=500)

# 3. Convert to DataFrame for easy ranking
df = pd.DataFrame.from_dict(results, orient='index').sort_values('f1', ascending=False)
df.index.name = 'Model'
df.reset_index(inplace=True)

print("Top 5 Models by F1-Score:")
display(df.head(10))

In [ ]:
# 4. Analyze results by threshold and language
df['lang'] = df['Model'].apply(lambda x: 'it' if '-it-' in x else 'en')
df['threshold'] = df['Model'].apply(lambda x: '060' if '-060' in x else '070' if '-070' in x else '080')

pivot = df.pivot_table(index=['lang', 'threshold'], values=['precision', 'recall', 'f1'], aggfunc='mean')
print("\nAverage Performance by Language and Threshold:")
display(pivot)